In [1]:
import math
from collections import Counter

import folium
import gurobipy as gurobi
import pandas as pd
from gurobipy import GRB

### Problem 1: Cell Tower Placement

Sets:

R = {1,...,25}: city regions
N(i): regions adjacent to region $i$

Parameters:

$a_{i,j} = 1$ if regions $i$ and $j$ are adjacent, else $0$

Decision variables:

$x_i = 1$ if a tower is installed in region $i$, else $0$

Objective (minimize towers installed):

$$\min \sum_{i \in R} x_i$$

Constraints (every region has a tower in it or in an adjacent region):

$$x_i + \sum_{j \in R} a_{i,j} x_j \ge 1 \quad \text{for all } i \in R$$

Binary restrictions:

$$x_i \in \{0,1\} \quad \text{for all } i \in R$$

In [2]:
adj = pd.read_csv("city_adjacency.csv", index_col=0)
adj.columns = adj.columns.astype(int)

regions = list(adj.index)
neighbors = {i: [j for j in regions if adj.loc[i, j] == 1] for i in regions}

m = gurobi.Model("cell_towers")

x = {i: m.addVar(vtype=GRB.BINARY, name=f"x_{i}") for i in regions}

m.setObjective(gurobi.quicksum(x[i] for i in regions), GRB.MINIMIZE)

for i in regions:
    m.addConstr(x[i] + gurobi.quicksum(x[j] for j in neighbors[i]) >= 1, name=f"cover_{i}")

m.optimize()

towers = [i for i in regions if x[i].X > 0.5]
print(f"\nMinimum number of towers: {int(round(m.ObjVal))}\n")
print(f"Towers installed in regions: {towers}\n")
for i in towers:
    print(f"  Tower in region {i:2d} covers regions {sorted([i] + neighbors[i])}")

Set parameter Username
Set parameter LicenseID to value 2859698
Academic license - for non-commercial use only - expires 2027-09-02
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 3 3200U with Radeon Vega Mobile Gfx, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Optimize a model with 25 rows, 25 columns and 107 nonzeros (Min)
Model fingerprint: 0xde2bae31
Model has 25 linear objective coefficients
Variable types: 0 continuous, 25 integer (25 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Found heuristic solution: objective 9.0000000
Presolve time: 0.02s
Presolved: 25 rows, 25 columns, 107 nonzeros
Variable types: 0 continuous, 25 integer (25 binary)

Root relaxation: objective 6.166667e+00, 35 iterations, 0.01 seconds (0.00 work units)

    Nodes  

### Problem 2: Storage Optimization for Moving Boxes, Revisited

Sets:

T = {1,...,15}: item types
B = {1,...,N}: available boxes, where $N$ is the FFD upper bound

Parameters:

$v_t$: volume of one item of type $t$
$q_t$: number of items of type $t$
$C = 50$: capacity of each box
$c = 30$: cost per box used

Decision variables:

$x_{t,b} \ge 0$, integer: number of items of type $t$ placed in box $b$
$y_b = 1$ if box $b$ is used, else $0$

Objective (minimize boxes used; total cost is $c$ times this value):

$$\min \sum_{b \in B} y_b$$

Constraints:

Every item of each type is packed:

$$\sum_{b \in B} x_{t,b} = q_t \quad \text{for all } t \in T$$

Box capacity is not exceeded, and a box holding any item must be counted as used:

$$\sum_{t \in T} v_t x_{t,b} \le C y_b \quad \text{for all } b \in B$$

Boxes are used in order (symmetry breaking, so Gurobi does not explore relabelings of the same packing):

$$y_b \ge y_{b+1} \quad \text{for all } b = 1,\dots,N-1$$

Nonnegativity and integrality:

$$x_{t,b} \ge 0 \text{ and integer}, \quad y_b \in \{0,1\}$$

In [3]:
items_df = pd.read_csv("items_data.csv")

vol = dict(zip(items_df["item_type"], items_df["volume"]))
qty = dict(zip(items_df["item_type"], items_df["quantity"]))
types = list(vol)
C = 50
COST = 30


def first_fit_decreasing(vol, qty, C):
    """Pack items largest-first into the first box with room. Returns a list of Counters (type -> count)."""
    packing, loads = [], []
    for t in sorted(vol, key=lambda t: vol[t], reverse=True):
        for _ in range(qty[t]):
            for k in range(len(packing)):
                if loads[k] + vol[t] <= C:
                    packing[k][t] += 1
                    loads[k] += vol[t]
                    break
            else:
                packing.append(Counter({t: 1}))
                loads.append(vol[t])
    return packing


ffd = first_fit_decreasing(vol, qty, C)
N = len(ffd)
total_volume = sum(vol[t] * qty[t] for t in types)

print(f"Total items: {sum(qty.values())}, total volume: {total_volume} cu ft")
print(f"Volume lower bound: ceil({total_volume}/{C}) = {math.ceil(total_volume / C)} boxes")
print(f"First-fit decreasing upper bound: N = {N} boxes\n")

boxes = list(range(1, N + 1))

m = gurobi.Model("moving_boxes_revisited")

x = {(t, b): m.addVar(vtype=GRB.INTEGER, lb=0, ub=C // vol[t], name=f"x_{t}_{b}") for t in types for b in boxes}
y = {b: m.addVar(vtype=GRB.BINARY, name=f"y_{b}") for b in boxes}

m.setObjective(gurobi.quicksum(y[b] for b in boxes), GRB.MINIMIZE)

for t in types:
    m.addConstr(gurobi.quicksum(x[(t, b)] for b in boxes) == qty[t], name=f"pack_{t}")
for b in boxes:
    m.addConstr(gurobi.quicksum(vol[t] * x[(t, b)] for t in types) <= C * y[b], name=f"cap_{b}")
for b in boxes[:-1]:
    m.addConstr(y[b] >= y[b + 1], name=f"order_{b}")

for b in boxes:
    y[b].Start = 1
    for t in types:
        x[(t, b)].Start = ffd[b - 1][t]

m.Params.TimeLimit = 600 
m.Params.MIPFocus = 1      

m.optimize()

status = "Optimal" if m.Status == GRB.OPTIMAL else "Time limit reached"
best = int(round(m.ObjVal))
print(f"\nStatus: {status}")
print(f"Boxes used in best solution found: {best}  (cost ${COST * best:,})")
print(f"Best bound found: {m.ObjBound:.2f} -> at least {math.ceil(m.ObjBound - 1e-6)} boxes")
print(f"MIP gap: {100 * m.MIPGap:.2f}%,  runtime: {m.Runtime:.1f} s\n")

patterns = Counter()
for b in boxes:
    if y[b].X > 0.5:
        contents = tuple((t, int(round(x[(t, b)].X))) for t in types if x[(t, b)].X > 0.5)
        patterns[contents] += 1

print(f"{'Boxes':>5}  {'Volume':>6}  Contents (type x count)")
for contents, count in patterns.most_common():
    load = sum(vol[t] * n for t, n in contents)
    desc = ", ".join(f"{t}x{n}" for t, n in contents)
    print(f"{count:>5}  {load:>6}  {desc}")

Total items: 1375, total volume: 21484 cu ft
Volume lower bound: ceil(21484/50) = 430 boxes
First-fit decreasing upper bound: N = 448 boxes

Set parameter TimeLimit to value 600
Set parameter MIPFocus to value 1
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 3 3200U with Radeon Vega Mobile Gfx, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Non-default parameters:
TimeLimit  600
MIPFocus  1

Optimize a model with 910 rows, 7168 columns and 14782 nonzeros (Min)
Model fingerprint: 0x62e86cc9
Model has 448 linear objective coefficients
Variable types: 0 continuous, 7168 integer (448 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 7e+00]
  RHS range        [2e+01, 2e+02]

Loaded user MIP start with objective 448

Presolve removed 99 rows and 99 columns
Presolve time: 0.18s
Presolved: 811 rows, 

### Problem 3: College Football Tournament Scheduling

Sets:

U = {1,...,50}: university teams
P = {(i,j) : i, j ∈ U, i < j}: possible matches (each unordered pair listed once)

Parameters:

$d_{i,j}$: haversine distance in miles between the cities of teams $i$ and $j$

Decision variables:

$x_{i,j} = 1$ if teams $i$ and $j$ play each other, else $0$, for all $(i,j) \in P$

Objective (minimize total travel distance; one team travels per match, so each match adds $d_{i,j}$ once):

$$\min \sum_{(i,j) \in P} d_{i,j}\, x_{i,j}$$

Constraints (each team plays exactly one match):

$$\sum_{j:\,(k,j) \in P} x_{k,j} + \sum_{i:\,(i,k) \in P} x_{i,k} = 1 \quad \text{for all } k \in U$$

Binary restrictions:

$$x_{i,j} \in \{0,1\} \quad \text{for all } (i,j) \in P$$

In [4]:
teams = pd.read_csv("tournament_teams.csv").set_index("id")


def haversine(lat1, lon1, lat2, lon2):
    """Great-circle distance in miles between two (latitude, longitude) points."""
    R = 3958.8 #Earth radius (miles)
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlam / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


U = list(teams.index)
pairs = [(i, j) for i in U for j in U if i < j]
dist = {
    (i, j): haversine(teams.loc[i, "latitude"], teams.loc[i, "longitude"],
                      teams.loc[j, "latitude"], teams.loc[j, "longitude"])
    for (i, j) in pairs
}

m = gurobi.Model("football_tournament")

x = {(i, j): m.addVar(vtype=GRB.BINARY, name=f"x_{i}_{j}") for (i, j) in pairs}

m.setObjective(gurobi.quicksum(dist[p] * x[p] for p in pairs), GRB.MINIMIZE)

for k in U:
    m.addConstr(gurobi.quicksum(x[(i, j)] for (i, j) in pairs if k in (i, j)) == 1, name=f"one_match_{k}")

m.optimize()

matches = [(i, j) for (i, j) in pairs if x[(i, j)].X > 0.5]
print(f"\nMinimum total travel distance: {m.ObjVal:,.1f} miles")
print(f"Time to solve: {m.Runtime:.2f} seconds\n")
for (i, j) in matches:
    print(f"  {teams.loc[i, 'university']} vs {teams.loc[j, 'university']}: {dist[(i, j)]:.1f} miles")

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 3 3200U with Radeon Vega Mobile Gfx, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Optimize a model with 50 rows, 1225 columns and 2450 nonzeros (Min)
Model fingerprint: 0x851b02fc
Model has 1225 linear objective coefficients
Variable types: 0 continuous, 1225 integer (1225 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e+01, 3e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Found heuristic solution: objective 26046.599621
Presolve time: 0.02s
Presolved: 50 rows, 1225 columns, 2450 nonzeros
Variable types: 0 continuous, 1225 integer (1225 binary)

Root relaxation: objective 4.154112e+03, 49 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    Be

In [5]:
tournament_map = folium.Map(location=[39.5, -98.35], zoom_start=4)

for (i, j) in matches:
    a, b = teams.loc[i], teams.loc[j]
    folium.PolyLine(
        [(a["latitude"], a["longitude"]), (b["latitude"], b["longitude"])],
        color="red", weight=3,
        tooltip=f"{a['university']} vs {b['university']} ({dist[(i, j)]:.0f} mi)",
    ).add_to(tournament_map)

for k in U:
    folium.CircleMarker(
        location=(teams.loc[k, "latitude"], teams.loc[k, "longitude"]),
        radius=5, color="navy", fill=True, fill_opacity=0.9,
        popup=f"{teams.loc[k, 'university']} ({teams.loc[k, 'city']}, {teams.loc[k, 'state']})",
    ).add_to(tournament_map)

tournament_map.save("tournament_pairings_map.html")
tournament_map